## Error distribution and statistics

In this section we look at the raw distribution and statistics of reconciled and base forecasts across all hierarchies.

In [ ]:
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root

import pandas as pd
import plotly.express as px
import python.utils as utils
import python.plots as plots
from python.time_series import BASE_FORECASTS, RECONCILED_FORECASTS, TimeSeriesType
from python.aggregation import MONTHS, FORTNIGHTS, WEEKS, WEEKDAYS, WEEKENDS, DAYS
from python.temporal_hierarchy import TemporalHierarchy, MinimalTemporalHierarchy, SemanticTemporalHierarchy
from python.error_metrics import ErrorMetrics, MergedErrorMetrics, MetricsIndex, LoadedErrorMetrics, ErrorDifference, FilteredErrorMetrics, ClonedErrorMetrics, CustomMetrics
from python.error_metrics import MetricsIndex as MI # Alias
from python.forecast_reconciliation import ForecastReconciliation

In [ ]:
em = LoadedErrorMetrics('metrics/base_hier_errors.csv')
inc = pd.read_csv('metrics/base_hier_inconsistency.csv', index_col=0)

From the processing of the errors, we now only have the 5 different reconciled forecasts, and the single base forecast. Now lets filter for NRMSE metrics specifically.

In [ ]:
nrmse_em = FilteredErrorMetrics(em, MI(error_metric='nrmse'))

Also lets define the base hierarchies just for easier aliasing.

In [ ]:
sth = SemanticTemporalHierarchy()
mth = MinimalTemporalHierarchy()
wth = TemporalHierarchy([MONTHS, WEEKS, DAYS], 'Minimal+Weeks')
wweth = TemporalHierarchy([MONTHS, WEEKS, WEEKDAYS, WEEKENDS, DAYS], 'Weekends+Weekdays')
wfth = TemporalHierarchy([MONTHS, FORTNIGHTS, WEEKS, DAYS], 'Weeks+Biweeks')

### Error statistics
First checking the statistics of the NRMSE errors across hierarchies in both daily and monthly granularity, comparing the base and reconciled performance
#### Days

In [ ]:
nrmse_em.get_error_stats(MI(granularity=DAYS))

Here we can see that at daily granularity, the median error across the different hierarchies are roughly the same, all of them being higher than the base forecast error median. The mean is single digit only at the base forecasts and at the minimal hierarchy, where we can see from the max value that no new significant outlier was introduced, unlike other hierarchies, which all have extreme outliers above the 90th percentile which make the mean and variance of the errors meaningless.

We can also note that `Minimal` hierarchy achieved the best median error after reconciliation.

#### Months

In [ ]:
nrmse_em.get_error_stats(MI(granularity=MONTHS))

At monthly granularity we can see that the medians across the different hierarchies are similar still, but they are all improvements compared to the base forecast error. We can see worsening of reconciled forecasts only at the 90th percentile. It is clear that outliers were present in the base forecasts as well, and in most hierarchies, except Minimal, they were all 'amplified'.

We can note that the `Semantic` hierarchy achieved the best median error after reconciliation. But it is also worth noting that `Minimal` lacks outliers at monthly granularity too.

### Error distributions
After checking the difference made to the errors across the hierarchies, let's check the raw distributions of the reconciled errors. 

#### Histograms
I will plot histograms of the distributions. I will remove outliers again for clarity.

In [ ]:
plots.plot_error_distribution(nrmse_em,
                              MI.indeces_from_product(granularities=[DAYS], hierarchies=[sth, mth, wth, wweth, wfth], forecast_types=[RECONCILED_FORECASTS]) + \
                              [MI(hierarchy=mth, granularity=DAYS, forecast_type=BASE_FORECASTS)],
                              outlier_threshold=0.99,
                              title_info='Daily granularity').show()

The distribution of errors on daily granularity look more or less the same. Though Minimal has the highest peak at 0-0.15. Semantic has the highest peak in the next step

In [ ]:
plots.plot_error_distribution(nrmse_em,
                              MI.indeces_from_product(granularities=[DAYS], hierarchies=[sth, mth, wth, wweth, wfth], forecast_types=[RECONCILED_FORECASTS]) + \
                              [MI(hierarchy=mth, granularity=DAYS, forecast_type=BASE_FORECASTS)],
                              outlier_threshold=0.99, bin_size=0.3,
                              title_info='Daily granularity - low resolution').show()

With lower resolution, it is clear Minimal+Weeks concentrates the most at low values, with Weekends + Weekdays being close second. Though the highest peak of lowest bin is with Weeks+Biweeks. Semantic does perform slightly worse.

In [ ]:
plots.plot_error_distribution(nrmse_em,
                              MI.indeces_from_product(granularities=[MONTHS], hierarchies=[sth, mth, wth, wweth, wfth], forecast_types=[RECONCILED_FORECASTS]) + \
                              [MI(hierarchy=mth, granularity=MONTHS, forecast_type=BASE_FORECASTS)],
                              outlier_threshold=0.99,
                              title_info='Monthly granularity').show()

On monthly level, Semantic outperforms Minimal, with higher skewness. Although it looks the same as any other hierarchy. So here visibly Minimal error performs bad.

In [ ]:
plots.plot_error_distribution(nrmse_em,
                              MI.indeces_from_product(granularities=[MONTHS], hierarchies=[sth, mth, wth, wweth, wfth], forecast_types=[RECONCILED_FORECASTS]) + \
                              [MI(hierarchy=mth, granularity=MONTHS, forecast_type=BASE_FORECASTS)],
                              outlier_threshold=0.99, bin_size=0.3,
                              title_info='Monthly granularity - low resolution').show()

In lower resolution it is clear that Semantic and Weekends + Weekdays perform the best, which two are almost identical

#### QQ diagrams

In this section I will look at various QQ plots to see correlations between errors of different features. First I will look at the relationship between the errors in the same hierarchy at the daily and monthly granularity level.

In [ ]:
for h in (sth, mth, wth, wweth, wfth):
    plots.plot_error_qq(nrmse_em,
                        MI(hierarchy=h, granularity=DAYS, forecast_type=RECONCILED_FORECASTS),
                        MI(hierarchy=h, granularity=MONTHS, forecast_type=RECONCILED_FORECASTS),
                        outlier_threshold=0.99).show()

All the hierarchies look rather similar, with matching error at the beginning, then the monthly errors slowly increasing. `Minimal` has the biggest gap from the identity line.

Now I will plot the QQ diagram of base forecast errors to reconciled errors per granularity and hierarchy. First the daily granularity

In [ ]:
for h in (sth, mth, wth, wweth, wfth):
    plots.plot_error_qq(nrmse_em, MI(hierarchy=mth, granularity=DAYS, forecast_type=BASE_FORECASTS),
                                  MI(hierarchy=h, granularity=DAYS, forecast_type=RECONCILED_FORECASTS),
                                  outlier_threshold=0.99).show()

All the distributions look smooth and regular. `Minimal` has the QQ plot closest to the identity line at daily granularity.

Now looking at monthly granularity. At quantiles below 0.25, there distributions are above the indentiy line meaning there are less very small errors after reconciliation at daily granularity.

In [ ]:
for h in (sth, mth, wth, wweth, wfth):
    plots.plot_error_qq(nrmse_em, MI(hierarchy=mth, granularity=MONTHS, forecast_type=BASE_FORECASTS),
                                  MI(hierarchy=h, granularity=MONTHS, forecast_type=RECONCILED_FORECASTS),
                                  outlier_threshold=0.97).show()

Here the quantiles of the reconciled forecasts monthly error increase in all cases at some point, in a very similar fashion, which is explainable by the outliers we have seen introduced.

#### 2D distribution plots
In this section I will plot 2D heatmaps to visualize the joint distributions of two errors at once. First let's look at the joint distribution of the base forecast errors at daily and monthly granularity, on a scatter plot and also on a heatmap.

In [ ]:
plots.plot_error_func(nrmse_em, MI(granularity=DAYS, forecast_type=BASE_FORECASTS),
                                MI(granularity=MONTHS, forecast_type=BASE_FORECASTS),
                      outlier_threshold=0.95).show()

In [ ]:
plots.plot_error_heatmap(nrmse_em, MI(granularity=DAYS, forecast_type=BASE_FORECASTS),
                                   MI(granularity=MONTHS, forecast_type=BASE_FORECASTS),
                                   bins=15, outlier_threshold=0.95).show()

We can see two slight lines of trends/density: one going from the origin in the direction of the identity line and one from the origin straight up. This means there are two main cases: when the daily error roughly matched the monthly, and when the daily error stayed low no matter the monthly.

Now let's look at the same plot but for each hierarchy's reconciled forecasts.

In [ ]:
for h in (sth, mth, wth, wweth, wfth):
    plots.plot_error_heatmap(nrmse_em, MI(granularity=DAYS, forecast_type=RECONCILED_FORECASTS, hierarchy=h),
                                       MI(granularity=MONTHS, forecast_type=RECONCILED_FORECASTS, hierarchy=h),
                                       bins=15, outlier_threshold=0.95).show()

Here, each hierarchy looks more or less the same, but in these reconciled cases (unlike the base forecast case) the only trend line visible is the identiy line, meaning the errors at the two granularities roughy match.

### Conclusions
As we saw in all subsections, the two prevelant extremes in all aspects were always `Semantic` and `Minimal` hierarchy. This is why in conclusions I will only mention traits of these hierarchies' reconciliations.

**Semantic**
- Pros
    - When looking at monthly granularity reconciled forecast errors, this hierarchy had the lowest median and other percentiles, across all hierarchies.
- Cons
    - When looking at daily granularity reconciled forecast errors, this hierarchy had the highest median, across all hierarchies.

**Minimal**
- Pros
    - When looking at daily granularity reconciled forecast errors, this hierarchy had the lowest median, across all hierarchies.
- Cons
    - When looking at daily granularity reconciled forecast errors, this hierarchy had the highest median, across all hierarchies.
    - When looking at QQ plots of the monthly vs. daily reconciled forecast errors, this hierarchy had the biggest gap from the identity line, signaling much bigger monthly errors than daily ones

Highlights:
- Daily/Monthly plain stats
- monthly vs. daily error QQ plots
- Error histograms
- Base forecast error heatmaps vs Reconciled forecast error heatmaps